Connect To Drive and Load the LLM wrapper

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from google.colab import userdata

PROJECT_ROOT = '/content/drive/MyDrive/Projects/multi-agent-discovery'
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))

sys.modules.pop('llm.gemini_client', None)
from llm.gemini_client import GeminiLLM
MODEL = "gemini-3.6-flash"
llm = GeminiLLM(api_key=userdata.get('GOOGLE_API_KEY'), model=MODEL)
print("llm now using:", llm.model)

Mounted at /content/drive
llm now using: gemini-3.6-flash


define the real shared state

In [ ]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/graph/state.py
"""Shared state for the multi-agent movie recommendation graph."""
from typing import TypedDict, Annotated
import operator


class AgentState(TypedDict):
    query: str
    plan: dict
    candidates: list
    critic_feedback: str
    critic_decision: str
    relax_state: dict
    verified: list
    recommendations: list
    iterations: int
    trajectory: Annotated[list, operator.add]

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/graph/state.py


define the Plan schema + Planner agent

In [ ]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/planner.py
"""Planner agent: natural-language query -> structured, validated Plan."""
from typing import Optional, Literal
from pydantic import BaseModel, Field

MOVIELENS_GENRES = ["Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
                    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
                    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western", "IMAX"]


class Plan(BaseModel):
    anchor_titles: list[str] = Field(default_factory=list,
        description="Exact movie titles the user named as references/anchors. Empty list if none.")
    mood: str = Field(description="The mood, tone, or themes the user wants, in a few words.")
    semantic_query: str = Field(
        description="A concise search phrase capturing plot/mood/theme for semantic search, "
                    "stripped of hard constraints like runtime or year.")
    strategy: Literal["cf", "semantic", "hybrid"] = Field(
        description="'cf' if the user only named anchor movies; 'semantic' if only a vibe/theme; "
                    "'hybrid' if both.")
    prefer_popular: bool = Field(
        description="True if the user wants well-known/popular films; "
                    "False for hidden gems / underrated / obscure.")
    genres_include: list[str] = Field(default_factory=list,
        description=f"Required genres, chosen ONLY from this list: {MOVIELENS_GENRES}. Empty if none.")
    runtime_max: Optional[int] = Field(default=None, description="Max runtime in minutes, or null.")
    runtime_min: Optional[int] = Field(default=None, description="Min runtime in minutes, or null.")
    year_min: Optional[int] = Field(default=None, description="Earliest release year, or null.")
    year_max: Optional[int] = Field(default=None, description="Latest release year, or null.")
    exclude_titles: list[str] = Field(default_factory=list,
        description="Movie titles the user wants excluded. Empty list if none.")


PLANNER_SYSTEM = (
    "You are the planning agent in a movie recommendation system. "
    "Decompose the user's request into a precise, structured plan for downstream tools. "
    f"For genres, use ONLY these exact labels: {MOVIELENS_GENRES}. "
    "Read 'underrated', 'obscure', 'hidden gem', 'deep cut', 'nothing mainstream' as prefer_popular=False; "
    "read 'popular', 'classic', 'well-known', 'famous', 'mainstream', 'nothing too obscure' as prefer_popular=True. "
    "If the user only names movies -> strategy='cf'; only describes a vibe -> strategy='semantic'; both -> strategy='hybrid'. "
    "IMPORTANT: semantic_query must describe ONLY plot, mood, tone, or themes for plot-based search. "
    "NEVER put runtime, year/decade, popularity words (underrated/obscure/popular/mainstream), or the words 'movie'/'film' in it. "
    "Example: for 'an underrated 80s horror, short' -> semantic_query='scary supernatural horror', NOT 'underrated obscure 80s horror movie'."
)


class Planner:
    def __init__(self, llm):
        self.llm = llm

    def plan(self, query: str) -> Plan:
        p = self.llm.structured(query, Plan, system=PLANNER_SYSTEM)
        valid = set(MOVIELENS_GENRES)
        p.genres_include = [g for g in p.genres_include if g in valid]   # defensive: drop invalid genres
        return p


def planner_node(state, planner: Planner):
    """LangGraph node. (Bound to a Planner instance via functools.partial when the graph is assembled.)"""
    p = planner.plan(state["query"])
    return {
        "plan": p.model_dump(),
        "trajectory": [f"PLANNER: strategy={p.strategy} | anchors={p.anchor_titles} | "
                       f"popular={p.prefer_popular} | genres={p.genres_include} | "
                       f"runtime<=({p.runtime_max}) | years=({p.year_min},{p.year_max}) | "
                       f"exclude={p.exclude_titles}"],
    }

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/planner.py


build the planner and stress-test it

In [ ]:
import sys
sys.modules.pop('agents.planner', None)
from agents.planner import Planner
planner = Planner(llm)

queries = [
    "something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure",
    "an underrated 80s horror movie, short, nothing mainstream",
    "feel-good animated family movies for kids",
    "a long epic war drama from the 90s, but not Saving Private Ryan",
    "twisty crime thrillers with unreliable narrators",
]
for q in queries:
    p = planner.plan(q)
    print("Q:", q)
    print("  ", p.model_dump(), "\n")

Q: something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure
   {'anchor_titles': ['Inception', 'The Matrix'], 'mood': 'accessible mind-bending sci-fi action', 'semantic_query': 'mind bending sci-fi action thriller about simulated reality', 'strategy': 'hybrid', 'prefer_popular': True, 'genres_include': ['Action', 'Sci-Fi'], 'runtime_max': 120, 'runtime_min': None, 'year_min': None, 'year_max': None, 'exclude_titles': []} 

Q: an underrated 80s horror movie, short, nothing mainstream
   {'anchor_titles': [], 'mood': 'scary, dark, atmospheric', 'semantic_query': 'scary atmospheric horror', 'strategy': 'semantic', 'prefer_popular': False, 'genres_include': ['Horror'], 'runtime_max': 90, 'runtime_min': None, 'year_min': 1980, 'year_max': 1989, 'exclude_titles': []} 

Q: feel-good animated family movies for kids
   {'anchor_titles': [], 'mood': 'feel-good', 'semantic_query': 'feel-good uplifting heartwarming family story', 'strategy': 'semantic', 'prefe

load all tools

In [5]:
!pip install implicit -q
!pip install -q faiss-cpu "sentence-transformers>=3.0.0" "transformers>=4.51.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.2 MB/s eta 0:00:00


In [6]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [7]:
import os, sys
sys.path.append(os.path.join(PROJECT_ROOT, 'src'))
for m in ['tools.tool_a_als','tools.tool_b_semantic','tools.tool_c_details','tools.tool_d_filter','tools.reranker']:
    sys.modules.pop(m, None)
from tools.tool_a_als import ToolA
from tools.tool_b_semantic import ToolB
from tools.tool_c_details import ToolC
from tools.tool_d_filter import ToolD
from tools.reranker import PopularityReranker
import torch

ALS_DIR    = os.path.join(PROJECT_ROOT, 'artifacts/als')
FAISS_DIR  = os.path.join(PROJECT_ROOT, 'artifacts/faiss')
CATALOG    = os.path.join(PROJECT_ROOT, 'data/processed/catalog/catalog.parquet')
MOVIES_CSV = os.path.join(PROJECT_ROOT, 'data/raw/ml-32m/movies.csv')

tool_a   = ToolA(ALS_DIR, MOVIES_CSV)                    # anchor fold-in only -> no user_item needed
tool_b   = ToolB(FAISS_DIR, CATALOG, device='cuda' if torch.cuda.is_available() else 'cpu')
tool_c   = ToolC(CATALOG, MOVIES_CSV)
tool_d   = ToolD(CATALOG, MOVIES_CSV)
reranker = PopularityReranker(CATALOG)
print("all tools loaded")

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

all tools loaded


write the Retriever

In [ ]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/retriever.py
"""Retriever agent: executes the plan via Tools A/B/C/D + re-ranker. Plan-driven, no LLM call."""
import re
import pandas as pd


class Retriever:
    def __init__(self, tool_a, tool_b, tool_c, tool_d, reranker, catalog_path,
                 cf_n=150, sem_n=50, sem_retrieve_n=400, pop_weight=0.15, candidate_cap=25):

        self.a, self.b, self.c, self.d, self.rr = tool_a, tool_b, tool_c, tool_d, reranker
        self.cf_n, self.sem_n, self.pop_weight, self.cap = cf_n, sem_n, pop_weight, candidate_cap
        self.sem_retrieve_n = sem_retrieve_n
        cat = pd.read_parquet(catalog_path)[["movieId", "title", "rating_count"]].copy()
        cat["norm"] = cat["title"].apply(self._norm)
        self._cat = cat

    @staticmethod
    def _norm(t):
        t = str(t).lower().strip()
        t = re.sub(r"\s*\(\d{4}\)\s*$", "", t)
        t = re.sub(r"^(the|a|an)\s+", "", t)
        return t.strip()

    def resolve_title(self, title):
        n = self._norm(title)
        hits = self._cat[self._cat["norm"] == n]
        if len(hits) == 0:
            hits = self._cat[self._cat["norm"].str.contains(re.escape(n), na=False)]
        if len(hits) == 0:
            return None
        return int(hits.sort_values("rating_count", ascending=False).iloc[0]["movieId"])

    def retrieve(self, plan):
        pool, log = {}, []

        # 1) CF via anchor fold-in
        if plan["strategy"] in ("cf", "hybrid") and plan["anchor_titles"]:
            anchor_ids = [mid for t in plan["anchor_titles"] if (mid := self.resolve_title(t))]
            if anchor_ids:
                cf = self.a.candidates_from_anchors(anchor_ids, n=self.cf_n)
                for mid in cf["candidates"]:
                    pool[mid] = {"source": "cf", "score": None}
                log.append(f"CF fold-in from {anchor_ids} -> {len(cf['candidates'])}")

        # 2) Semantic (+ popularity re-rank iff prefer_popular)
        if plan["strategy"] in ("semantic", "hybrid"):
            sem = self.b.semantic_search(plan["semantic_query"], n=self.sem_retrieve_n)["results"]
            #                                                      ^^^^^^^^^^^^^^^^^^^  <-- CHANGE 2: was self.sem_n
            if plan["prefer_popular"]:
                sem = self.rr.rerank(sem, pop_weight=self.pop_weight, n=self.sem_retrieve_n)
                #                                                       ^^^^^^^^^^^^^^^^^^^  <-- CHANGE 2: was self.sem_n
                log.append(f"semantic '{plan['semantic_query']}' + pop re-rank -> {len(sem)}")
            else:
                log.append(f"semantic '{plan['semantic_query']}' (hidden-gems, no re-rank) -> {len(sem)}")
            for r in sem:
                sc = r.get("blended_score", r["score"])
                if r["movieId"] in pool:
                    pool[r["movieId"]]["source"] += "+semantic"
                else:
                    pool[r["movieId"]] = {"source": "semantic", "score": sc}

        # 3) Hard constraints (Tool D)
        cand_ids = list(pool.keys())
        exclude_ids = [mid for t in plan["exclude_titles"] if (mid := self.resolve_title(t))]
        yr = (plan["year_min"], plan["year_max"]) if (plan["year_min"] or plan["year_max"]) else None
        filt = self.d.filter_by(cand_ids, runtime_max=plan["runtime_max"], runtime_min=plan["runtime_min"],
                                genre_in=plan["genres_include"] or None, year_range=yr, exclude_ids=exclude_ids)
        log.append(f"filter {filt['n_in']} -> {filt['n_out']}")

        # 4) Cap + enrich with metadata (Tool C)
        enriched = []
        for mid in filt["filtered"][:self.cap]:
            det = self.c.get_movie_details(mid)
            if det.get("found"):
                det["source"] = pool[mid]["source"]
                det["retrieval_score"] = pool[mid]["score"]
                enriched.append(det)
        log.append(f"enriched {len(enriched)} candidates")
        return enriched, log


def retriever_node(state, retriever):
    cands, log = retriever.retrieve(state["plan"])
    return {"candidates": cands, "trajectory": ["RETRIEVER: " + " | ".join(log)]}

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/retriever.py


test the resolver, then the full retriever

In [ ]:
for m in ['agents.planner','agents.retriever']: sys.modules.pop(m, None)
from agents.planner import Planner
from agents.retriever import Retriever
planner   = Planner(llm)
retriever = Retriever(tool_a, tool_b, tool_c, tool_d, reranker, CATALOG)

# resolver sanity
for t in ["Inception", "The Matrix", "Lion King", "star wars"]:
    print(f"  resolve('{t}') -> {retriever.resolve_title(t)}")

# full retrieve on three query types
for q in ["something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure",
          "feel-good animated family movies for kids",
          "an underrated 80s horror movie, short, nothing mainstream"]:
    plan = planner.plan(q).model_dump()
    cands, log = retriever.retrieve(plan)
    print("\nQ:", q)
    for l in log: print("   -", l)
    print("   TOP:", [f"{c['title']} ({c['year']})" for c in cands[:8]])

  resolve('Inception') -> 79132
  resolve('The Matrix') -> 2571
  resolve('Lion King') -> 364
  resolve('star wars') -> 260

Q: something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure
   - CF fold-in from [79132, 2571] -> 150
   - semantic 'high-stakes reality-bending science fiction action' + pop re-rank -> 400
   - filter 547 -> 283
   - enriched 25 candidates
   TOP: ['WALL·E (2008)', 'Eternal Sunshine of the Spotless Mind (2004)', 'Raiders of the Lost Ark (1981)', 'Back to the Future (1985)', 'Men in Black (1997)', 'Blade Runner (1982)', 'Arrival (2016)', 'Ex Machina (2015)']

Q: feel-good animated family movies for kids
   - semantic 'uplifting heartwarming animated stories for children' + pop re-rank -> 400
   - filter 400 -> 272
   - enriched 25 candidates
   TOP: ['The Star (2017)', 'Ramona and Beezus (2010)', 'Inside Out (2015)', 'Chicken Run (2000)', 'Care Bears Movie II: A New Generation (1986)', 'Incredibles 2 (2018)', 'Raven the Littl

The Critic Agent

In [2]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/critic.py
"""Critic agent: verifies thematic fit (LLM), and drives graduated, alternating constraint relaxation."""
from pydantic import BaseModel, Field

# ---- relaxation caps ----
RUNTIME_STEP = 20          # minutes per runtime relaxation step
RUNTIME_MAX_STEPS = 3      # 3 x 20 = up to +60 min (e.g. 120 -> 180)
YEAR_STEP = 5              # years per side per year relaxation step
YEAR_MAX_STEPS = 4         # 4 x 5 = up to +/-20 years

def _relaxed_to_cap(relax_state):
    """True if any relaxable axis was pushed to its cap (=> results are a 'stretch')."""
    if not relax_state:
        return False
    return (relax_state.get("runtime_steps", 0) >= RUNTIME_MAX_STEPS or
            relax_state.get("year_steps", 0) >= YEAR_MAX_STEPS)

class Verdict(BaseModel):
    movieId: int
    keep: bool = Field(description="true if this movie's plot/tone matches the desired mood/theme")
    reason: str = Field(description="brief justification (one short phrase)")

class CriticVerdicts(BaseModel):
    verdicts: list[Verdict]


CRITIC_SYSTEM = (
    "You are the critic agent in a movie recommendation system. "
    "Given the user's desired mood/theme and candidate movies with plot overviews, "
    "judge whether EACH candidate genuinely matches the requested mood/theme. "
    "Hard constraints (runtime, genre, year) are ALREADY satisfied — do NOT re-check those. "
    "Judge ONLY thematic and tonal fit. Keep a movie if its plot/tone fits the request; drop it if it clearly does not. "
    "Be discerning but fair — do not drop a candidate that reasonably fits."
)


def _has_runtime(plan):
    return plan.get("runtime_max") is not None or plan.get("runtime_min") is not None

def _has_year(plan):
    return plan.get("year_min") is not None or plan.get("year_max") is not None


def relax_plan(plan, relax_state):
    """Relax ONE present constraint, alternating runtime<->year, each capped independently.
    genre and exclusions are NEVER relaxed. Returns (new_plan, new_relax_state, change_msg_or_None)."""
    rs = dict(relax_state) if relax_state else {"runtime_steps": 0, "year_steps": 0, "next": "runtime"}
    p = dict(plan)

    runtime_avail = _has_runtime(p) and rs["runtime_steps"] < RUNTIME_MAX_STEPS
    year_avail    = _has_year(p)    and rs["year_steps"]    < YEAR_MAX_STEPS

    if not runtime_avail and not year_avail:
        return p, rs, None                          # everything relaxable is exhausted -> caller reports "no match"

    # pick side: honor alternation, but skip a side that's unavailable
    side = rs["next"]
    if side == "runtime" and not runtime_avail: side = "year"
    if side == "year"    and not year_avail:    side = "runtime"

    if side == "runtime":
        if p.get("runtime_max") is not None: p["runtime_max"] += RUNTIME_STEP
        if p.get("runtime_min") is not None: p["runtime_min"] = max(0, p["runtime_min"] - RUNTIME_STEP)
        rs["runtime_steps"] += 1
        rs["next"] = "year"
        msg = f"runtime by {RUNTIME_STEP}min (now max={p.get('runtime_max')}, min={p.get('runtime_min')})"
    else:
        if p.get("year_min") is not None: p["year_min"] -= YEAR_STEP
        if p.get("year_max") is not None: p["year_max"] += YEAR_STEP
        rs["year_steps"] += 1
        rs["next"] = "runtime"
        msg = f"year window by +/-{YEAR_STEP} (now {p.get('year_min')}-{p.get('year_max')})"
    return p, rs, msg


class Critic:
    def __init__(self, llm, min_verified=5):
        self.llm = llm
        self.min_verified = min_verified

    def verify(self, plan, candidates):
        if not candidates:
            return [], []
        listing = "\n".join(
            f"- id={c['movieId']} | {c['title']} ({c.get('year')}) | {(c.get('overview') or '')[:200]}"
            for c in candidates)
        prompt = (f"Desired mood/theme: {plan['mood']}\n"
                  f"Semantic intent: {plan['semantic_query']}\n\n"
                  f"Candidates:\n{listing}\n\n"
                  f"For EACH candidate id, decide keep or drop (thematic/tonal fit only), with a brief reason.")
        result = self.llm.structured(prompt, CriticVerdicts, system=CRITIC_SYSTEM)
        vmap = {v.movieId: v.keep for v in result.verdicts}
        verified = [c for c in candidates if vmap.get(c["movieId"], True)]
        return verified, result.verdicts


def critic_node(state, critic):
    plan, cands = state["plan"], state["candidates"]
    relax_state = state.get("relax_state") or {"runtime_steps": 0, "year_steps": 0, "next": "runtime"}
    verified, _ = critic.verify(plan, cands)
    it = state["iterations"] + 1
    kept = len(verified)
    log = [f"CRITIC: {kept}/{len(cands)} candidates match the mood (iteration {it})"]

    # enough matches -> accept
    if kept >= critic.min_verified:
        applied = relax_state.get("applied", [])
        if applied and _relaxed_to_cap(relax_state):
            fb = ("STRETCH: No movies truly matched all your constraints. "
                  "Here are the closest matches, found only after loosening: " + "; ".join(applied))
            tag = "STRETCH"
        elif applied:
            fb = "Relaxed constraints to find matches: " + "; ".join(applied)
            tag = f"ACCEPT (relaxed)"
        else:
            fb = ""
            tag = "ACCEPT"
        return {"verified": verified, "iterations": it, "critic_decision": "accept",
                "relax_state": relax_state, "critic_feedback": fb,
                "trajectory": log + [f"CRITIC: {tag} ({kept} verified)"]}

    # not enough -> try to relax one present constraint
    new_plan, new_rs, change = relax_plan(plan, relax_state)
    if change is None:
        # nothing left to relax -> stop, honest "no full match" (keep whatever matched, if any)
        applied = relax_state.get("applied", [])
        return {"verified": verified, "iterations": it, "critic_decision": "accept",
                "relax_state": relax_state,
                "critic_feedback": "NO_FULL_MATCH",
                "trajectory": log + [f"CRITIC: STOP — no movies matched all constraints after relaxing {applied or 'nothing available'}"]}

    new_rs["applied"] = relax_state.get("applied", []) + [change]
    return {"verified": verified, "iterations": it, "critic_decision": "retry",
            "plan": new_plan, "relax_state": new_rs,
            "critic_feedback": f"only {kept} matched; relaxed {change}",
            "trajectory": log + [f"CRITIC: RETRY — relaxed {change}"]}


def route_after_critic(state):
    return "explain" if state["critic_decision"] == "accept" else "retrieve"

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/critic.py


Verification Demo

In [ ]:
for m in ['graph.state','agents.planner','agents.retriever','agents.critic']:
    sys.modules.pop(m, None)
from agents.planner import Planner
from agents.retriever import Retriever
from agents.critic import Critic, relax_plan, critic_node, route_after_critic   # <-- added the two functions
planner   = Planner(llm)
retriever = Retriever(tool_a, tool_b, tool_c, tool_d, reranker, CATALOG)
critic    = Critic(llm, min_verified=5, max_iterations=2)

q = "twisty crime thrillers with unreliable narrators"
plan = planner.plan(q).model_dump()
cands, _ = retriever.retrieve(plan)
verified, verdicts = critic.verify(plan, cands)

print(f"Q: {q}\nmood: {plan['mood']}\n")
print(f"{len(verified)}/{len(cands)} candidates kept. Per-candidate verdicts:\n")
title_by_id = {c['movieId']: c['title'] for c in cands}
for v in verdicts:
    mark = "KEEP" if v.keep else "DROP"
    print(f"  [{mark}] {title_by_id.get(v.movieId, v.movieId)} — {v.reason}")

Q: twisty crime thrillers with unreliable narrators
mood: twisty, psychological, deceptive

15/25 candidates kept. Per-candidate verdicts:

  [KEEP] Murder, My Sweet — classic noir filled with mystery and deceit
  [KEEP] Secret — deceptive crime investigation with hidden secrets
  [KEEP] The Big Sleep — complex noir web of blackmail and deceit
  [KEEP] Deceiver — intense psychological mind games and interrogation
  [KEEP] Lost Highway — surreal, psychological, and deceptive identity puzzle
  [DROP] Jack Reacher — straightforward action-mystery rather than psychological deception
  [KEEP] The Salton Sea — twisty neo-noir where nothing is as it seems
  [KEEP] Police Python 357 — crime thriller centered on a web of deceit and framing
  [DROP] Chasing Ghosts — standard police procedural lacking psychological depth
  [DROP] Spectre — blockbuster spy action rather than twisty psychological thriller
  [DROP] Pride and Glory — police corruption drama without a psychological twisty tone
  [DROP

The Accept vs Retry Decision, and The Relaxation

In [ ]:
# Normal critic (min_verified=5) on a good pool -> should ACCEPT
out_accept = critic_node({"plan": plan, "candidates": cands, "iterations": 0}, critic)
print("decision:", out_accept["critic_decision"], "| verified:", len(out_accept["verified"]))
for t in out_accept["trajectory"]: print("  ", t)

# Force the RETRY path to see relaxation: a critic that demands more than the cap can give
strict = Critic(llm, min_verified=30, max_iterations=2)     # 30 > candidate_cap(25) -> can't be met -> retry
out_retry = critic_node({"plan": plan, "candidates": cands, "iterations": 0}, strict)
print("\ndecision:", out_retry["critic_decision"], "| feedback:", out_retry["critic_feedback"])
print("plan runtime_max before:", plan.get("runtime_max"), "-> after:", out_retry["plan"].get("runtime_max"))

decision: accept | verified: 15
   CRITIC: 15/25 candidates match the mood (iteration 1)
   CRITIC: ACCEPT (15 verified)

decision: retry | feedback: only 14 matched; relaxed constraint: kept only the primary genre
plan runtime_max before: None -> after: None


Manual one-iteration loop

In [ ]:
# Simulate: retrieve -> critic(retry, relax) -> retrieve(relaxed) -> critic(accept)
q2 = "an underrated 80s horror movie, short, nothing mainstream"
plan2 = planner.plan(q2).model_dump()
cands2, _ = retriever.retrieve(plan2)
print(f"iter 0: {len(cands2)} candidates (runtime_max={plan2['runtime_max']}, years={plan2['year_min']}-{plan2['year_max']})")

strict = Critic(llm, min_verified=15, max_iterations=3)
step = critic_node({"plan": plan2, "candidates": cands2, "iterations": 0}, strict)
print(f"iter 1 critic: {step['critic_decision']} — {step.get('critic_feedback','')}")

if step["critic_decision"] == "retry":
    cands3, _ = retriever.retrieve(step["plan"])              # retrieve with the RELAXED plan
    print(f"iter 1 retrieve (relaxed): {len(cands3)} candidates "
          f"(runtime_max={step['plan']['runtime_max']})")

iter 0: 14 candidates (runtime_max=90, years=1980-1989)
iter 1 critic: retry — only 5 matched; relaxed constraint: dropped runtime_max
iter 1 retrieve (relaxed): 22 candidates (runtime_max=None)


The Explainer Agent

In [ ]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/explainer.py
"""Explainer agent: verified candidates -> ranked recommendations with grounded, cited explanations."""
from pydantic import BaseModel, Field


class Recommendation(BaseModel):
    movieId: int = Field(description="movieId chosen ONLY from the provided verified candidates")
    explanation: str = Field(
        description="2-3 sentences on why it fits, citing specific plot elements from THIS movie's overview "
                    "and (if anchors are given) its thematic link to them")

class Recommendations(BaseModel):
    ranked: list[Recommendation]


EXPLAINER_SYSTEM = (
    "You are the explainer agent in a movie recommendation system. "
    "From the VERIFIED candidates provided (each with a plot overview), select and rank the best matches "
    "for the user's request, best first. "
    "For each pick, write 2-3 sentences that are STRICTLY GROUNDED in that movie's provided overview — "
    "cite specific plot elements, characters, or themes that actually appear in it. "
    "If anchor movies are named, draw an explicit thematic link to them. "
    "RULES: never invent plot details not in the overview; never recommend a movie not in the provided list."
)


class Explainer:
    def __init__(self, llm, top_n=5):
        self.llm = llm
        self.top_n = top_n

    def explain(self, plan, verified):
        if not verified:
            return []
        listing = "\n".join(
            f"- id={c['movieId']} | {c['title']} ({c.get('year')}) | overview: {(c.get('overview') or '')[:300]}"
            for c in verified)
        anchors = plan.get("anchor_titles") or []
        prompt = (
            f"User wants: {plan['mood']}\n"
            f"Semantic intent: {plan['semantic_query']}\n"
            f"Anchor movies (for thematic linking): {anchors if anchors else 'none'}\n\n"
            f"Verified candidates:\n{listing}\n\n"
            f"Select and rank the top {self.top_n}. Give each a grounded 2-3 sentence explanation.")
        result = self.llm.structured(prompt, Recommendations, system=EXPLAINER_SYSTEM)

        by_id = {c['movieId']: c for c in verified}
        recs = []
        for r in result.ranked:
            if r.movieId in by_id:                       # defensive: only real verified movies survive
                c = by_id[r.movieId]
                recs.append({"movieId": r.movieId, "title": c['title'], "year": c.get('year'),
                             "genres": c.get('genres'), "explanation": r.explanation})
        return recs[:self.top_n]


def explainer_node(state, explainer):
    recs = explainer.explain(state["plan"], state["verified"])
    return {"recommendations": recs,
            "trajectory": [f"EXPLAINER: produced {len(recs)} grounded recommendations"]}

Writing /content/drive/MyDrive/Projects/multi-agent-discovery/src/agents/explainer.py


Full Pipeline Test

In [ ]:
for m in ['agents.planner','agents.retriever','agents.critic','agents.explainer']:
    sys.modules.pop(m, None)
from agents.planner import Planner
from agents.retriever import Retriever
from agents.critic import Critic
from agents.explainer import Explainer
planner   = Planner(llm)
retriever = Retriever(tool_a, tool_b, tool_c, tool_d, reranker, CATALOG)
critic    = Critic(llm, min_verified=5, max_iterations=2)
explainer = Explainer(llm, top_n=5)

q = "something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure"
plan = planner.plan(q).model_dump()
cands, _    = retriever.retrieve(plan)
verified, _ = critic.verify(plan, cands)
recs        = explainer.explain(plan, verified)

print("Q:", q, "\n")
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['title']} ({r['year']})  [{'|'.join(r['genres'] or [])}]")
    print(f"   {r['explanation']}\n")

Q: something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure 

1. Total Recall (1990)  [Action|Adventure|Sci-Fi|Thriller]
   When construction worker Douglas Quaid visits Rekall to buy manufactured memories of Mars, a botched procedure forces him to question what is reality and what is not. Much like The Matrix and Inception, it uses simulated experiences to drive an exhilarating sci-fi action storyline. The premise stays direct and easy to follow while questioning the nature of truth.

2. Source Code (2011)  [Action|Drama|Mystery|Sci-Fi|Thriller]
   Decorated soldier Captain Colter Stevens finds himself waking up in an unknown man's body to catch a Chicago commuter train bomber. Similar to the structured, mind-bending reality jumps in Inception, the story maintains a clear objective across its sci-fi premise. The focused investigation offers fast-paced action without confusing the viewer.

3. Edge of Tomorrow (2014)  [Action|Sci-Fi|IMAX]
   Demoted

Grounding Spot-Check

In [ ]:
top = recs[0]
overview = next(c['overview'] for c in verified if c['movieId'] == top['movieId'])
print("RECOMMENDATION:", top['title'])
print("EXPLANATION:", top['explanation'])
print("\nSOURCE OVERVIEW:", overview)

RECOMMENDATION: Total Recall
EXPLANATION: When construction worker Douglas Quaid visits Rekall to buy manufactured memories of Mars, a botched procedure forces him to question what is reality and what is not. Much like The Matrix and Inception, it uses simulated experiences to drive an exhilarating sci-fi action storyline. The premise stays direct and easy to follow while questioning the nature of truth.

SOURCE OVERVIEW: Construction worker Douglas Quaid's obsession with the planet Mars leads him to visit Rekall, a virtual vacation company that manufactures memories. When something goes wrong during Quaid's memory implant procedure, his life turns upside down, leading him to question what is reality and what isn't.


The Graph Builder

In [3]:
%%writefile /content/drive/MyDrive/Projects/multi-agent-discovery/src/graph/build.py
"""Assembles the four agents into the full LangGraph state machine."""
from langgraph.graph import StateGraph, START, END

from graph.state import AgentState
from agents.planner import Planner, planner_node
from agents.retriever import Retriever, retriever_node
from agents.critic import Critic, critic_node, route_after_critic
from agents.explainer import Explainer, explainer_node


def build_graph(llm, tool_a, tool_b, tool_c, tool_d, reranker, catalog_path,
                min_verified=5, max_iterations=2, top_n=5):
    planner   = Planner(llm)
    retriever = Retriever(tool_a, tool_b, tool_c, tool_d, reranker, catalog_path)
    critic    = Critic(llm, min_verified=min_verified)
    explainer = Explainer(llm, top_n=top_n)

    g = StateGraph(AgentState)
    # bind each agent to its node via a closure (node sees only `state`; agent captured from scope)
    g.add_node("planner",   lambda s: planner_node(s, planner))
    g.add_node("retriever", lambda s: retriever_node(s, retriever))
    g.add_node("critic",    lambda s: critic_node(s, critic))
    g.add_node("explainer", lambda s: explainer_node(s, explainer))

    g.add_edge(START, "planner")
    g.add_edge("planner", "retriever")
    g.add_edge("retriever", "critic")
    g.add_conditional_edges("critic", route_after_critic,
                            {"retrieve": "retriever", "explain": "explainer"})   # critic's decision routes
    g.add_edge("explainer", END)
    return g.compile()

Overwriting /content/drive/MyDrive/Projects/multi-agent-discovery/src/graph/build.py


Build and  Visualize the Real Graph (APP)

In [8]:
for m in ['graph.state','graph.build','agents.planner','agents.retriever','agents.critic','agents.explainer']:
    sys.modules.pop(m, None)
from graph.build import build_graph

app = build_graph(llm, tool_a, tool_b, tool_c, tool_d, reranker, CATALOG)
print("graph compiled\n")
print(app.get_graph().draw_mermaid())

graph compiled

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	retriever(retriever)
	critic(critic)
	explainer(explainer)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	critic -. &nbsp;explain&nbsp; .-> explainer;
	critic -. &nbsp;retrieve&nbsp; .-> retriever;
	planner --> retriever;
	retriever --> critic;
	explainer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



The First Full End-To-End Run

In [ ]:
q = "something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure"
final = app.invoke({"query": q, "iterations": 0, "trajectory": [],"relax_state": {"runtime_steps": 0, "year_steps": 0, "next": "runtime"}})

print("QUERY:", q, "\n=== TRAJECTORY ===")
for t in final["trajectory"]:
  print("  ", t)
print("\ncritic_feedback:", final.get("critic_feedback"))
print("\n=== RECOMMENDATIONS ===")
for i, r in enumerate(final["recommendations"], 1):
    print(f"{i}. {r['title']} ({r['year']})\n   {r['explanation']}\n")

QUERY: something like Inception and The Matrix but less confusing, under 2 hours, nothing too obscure 
=== TRAJECTORY ===
   PLANNER: strategy=hybrid | anchors=['Inception', 'The Matrix'] | popular=True | genres=['Sci-Fi', 'Action'] | runtime<=(120) | years=(None,None) | exclude=[]
   RETRIEVER: CF fold-in from [79132, 2571] -> 150 | semantic 'sci-fi action simulated reality virtual world technology' + pop re-rank -> 400 | filter 546 -> 321 | enriched 25 candidates
   CRITIC: 7/25 candidates match the mood (iteration 1)
   CRITIC: ACCEPT (7 verified)
   EXPLAINER: produced 5 grounded recommendations

critic_feedback: 

=== RECOMMENDATIONS ===
1. Total Recall (1990)
   When construction worker Douglas Quaid visits Rekall, a company that manufactures memories, a botched implant procedure forces him to question what is reality and what isn't. Echoing the themes of 'Inception' and 'The Matrix', this action-packed sci-fi story revolves around simulated perception and high-stakes identity qu

Stream The Pipeline

In [ ]:
print("STREAMING the agent pipeline:\n")
for chunk in app.stream({"query": q, "iterations": 0, "trajectory": []}):
    for node, update in chunk.items():
        traj = update.get("trajectory", [])
        print(f"[{node}] {traj[-1] if traj else ''}")

STREAMING the agent pipeline:

[planner] PLANNER: strategy=hybrid | anchors=['Inception', 'The Matrix'] | popular=True | genres=['Sci-Fi', 'Action'] | runtime<=(120) | years=(None,None) | exclude=[]
[retriever] RETRIEVER: CF fold-in from [79132, 2571] -> 150 | semantic 'reality warping virtual reality sci-fi action straightforward plot' + pop re-rank -> 400 | filter 547 -> 251 | enriched 25 candidates
[critic] CRITIC: RETRY — relaxed plan (dropped runtime_max)
[retriever] RETRIEVER: CF fold-in from [79132, 2571] -> 150 | semantic 'reality warping virtual reality sci-fi action straightforward plot' + pop re-rank -> 400 | filter 547 -> 319 | enriched 25 candidates
[critic] CRITIC: ACCEPT (4 verified)
[explainer] EXPLAINER: produced 4 grounded recommendations


A Hidden-Gems Query

In [ ]:
q2 = "an underrated 80s horror movie, short, nothing mainstream"
final2 = app.invoke({"query": q2, "iterations": 0, "trajectory": []})
print("QUERY:", q2, "\n=== TRAJECTORY ===")
for t in final2["trajectory"]: print("  ", t)
print("\n=== RECOMMENDATIONS ===")
for i, r in enumerate(final2["recommendations"], 1):
    print(f"{i}. {r['title']} ({r['year']})\n   {r['explanation']}\n")

QUERY: an underrated 80s horror movie, short, nothing mainstream 
=== TRAJECTORY ===
   PLANNER: strategy=semantic | anchors=[] | popular=False | genres=['Horror'] | runtime<=(90) | years=(1980,1989) | exclude=[]
   RETRIEVER: semantic 'scary supernatural horror, creepy atmospheric slasher' (hidden-gems, no re-rank) -> 400 | filter 400 -> 14 | enriched 14 candidates
   CRITIC: 11/14 candidates match the mood (iteration 1)
   CRITIC: ACCEPT (11 verified)
   EXPLAINER: produced 5 grounded recommendations

=== RECOMMENDATIONS ===
1. Laurin (1989)
   This atmospheric 1980s horror film is set in a small 19th-century port town where children are mysteriously disappearing. It delivers a creepy, supernatural vibe as nine-year-old Laurin suffers from terrifying dreams and hallucinations of a man in black who stalks the town.

2. Dolls (1987)
   This retro horror film establishes a creepy, atmospheric setting inside a dark, haunted mansion. The story follows an eclectic group of guests, includin

Checking And Verifying The New Relaxation Plan (with no LLM calls)

In [ ]:
sys.modules.pop('agents.critic', None)
from agents.critic import relax_plan

# A plan with BOTH runtime and year constraints -> should alternate runtime<->year, each capped
plan = {"runtime_max": 120, "runtime_min": None, "year_min": 1980, "year_max": 1989,
        "genres_include": ["Horror"], "exclude_titles": ["Some Movie"]}
rs = {"runtime_steps": 0, "year_steps": 0, "next": "runtime"}

print("Alternating relaxation ladder:\n")
for step in range(1, 9):
    plan, rs, change = relax_plan(plan, rs)
    if change is None:
        print(f"  step {step}: EXHAUSTED — stop (no further relaxation)")
        break
    print(f"  step {step}: relaxed {change}  | genres still {plan['genres_include']} | exclude still {plan['exclude_titles']}")

Alternating relaxation ladder:

  step 1: relaxed runtime by 20min (now max=140, min=None)  | genres still ['Horror'] | exclude still ['Some Movie']
  step 2: relaxed year window by +/-5 (now 1975-1994)  | genres still ['Horror'] | exclude still ['Some Movie']
  step 3: relaxed runtime by 20min (now max=160, min=None)  | genres still ['Horror'] | exclude still ['Some Movie']
  step 4: relaxed year window by +/-5 (now 1970-1999)  | genres still ['Horror'] | exclude still ['Some Movie']
  step 5: relaxed runtime by 20min (now max=180, min=None)  | genres still ['Horror'] | exclude still ['Some Movie']
  step 6: relaxed year window by +/-5 (now 1965-2004)  | genres still ['Horror'] | exclude still ['Some Movie']
  step 7: relaxed year window by +/-5 (now 1960-2009)  | genres still ['Horror'] | exclude still ['Some Movie']
  step 8: EXHAUSTED — stop (no further relaxation)
